In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd drive

/content/drive


In [3]:
cd My Drive

/content/drive/My Drive


In [5]:
pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 24.0 MB/s  0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mne]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [7]:
# ============================================================
# DeepNeuroTransFormer
# 10-Fold Cross-Validation (SEGMENT-WISE, NOT SUBJECT-WISE)
# Sweep over NUMBER OF ATTENTION HEADS and NUMBER OF STACKED
# TRANSFORMER BLOCKS, comparing overall classification
# performance AND INFERENCE TIME across configurations.
#
# Dataset:
#   19 EEG channels
#   Sampling frequency = 250 Hz
#   Segment duration = 10 seconds
#   Samples per segment = 2500
#
# Input shape:
#   (2500, 19)
#
# Validation:
#   Simple 10-Fold Stratified Cross-Validation (StratifiedKFold)
#   Segments are shuffled and split independently of
#   participant identity. Segments from the SAME participant
#   can therefore appear in both the training and the test
#   set of a fold.
#
#   NOTE: this is the standard "segment-wise" protocol used in
#   many EEG papers. It usually gives much higher accuracy than
#   subject-wise CV, because the model can exploit
#   participant-specific characteristics. Results should not be
#   interpreted as generalisation to unseen participants.
#
# Normalisation:
#   StandardScaler is still fitted ONLY on the training
#   segments of each fold.
#
# Inference time:
#   Only the model forward pass is timed. EDF loading,
#   band-pass filtering, segmentation, per-fold
#   standardisation and array slicing are all completed
#   BEFORE the timers start and are NOT included.
#
# WARNING:
#   Running several configurations means running several full
#   10-fold cross-validations (i.e. training many models).
#   This can take a long time. Trim CONFIGURATIONS,
#   N_SPLITS, or EPOCHS below if you need a quicker pass.
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import gc
import time
import random
import warnings

import numpy as np
import pandas as pd
import mne
import tensorflow as tf

from scipy.signal import butter, filtfilt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

from tensorflow.keras.layers import (
    Input,
    Layer,
    MultiHeadAttention,
    LayerNormalization,
    Dense,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model


warnings.filterwarnings("ignore")


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

np.random.seed(SEED)
random.seed(SEED)

tf.random.set_seed(SEED)


# ============================================================
# 3. DATASET PARAMETERS
# ============================================================

# Change this path to the folder containing your EDF files
folder_path = "dataverse"

# EEG parameters
SFREQ = 250

LOWCUT = 0.5
HIGHCUT = 50.0

FILTER_ORDER = 2

# 10-second non-overlapping segments
SEGMENT_LENGTH = 10

SAMPLES_PER_SEGMENT = SEGMENT_LENGTH * SFREQ

# 10 x 250 = 2500
print("Samples per segment:", SAMPLES_PER_SEGMENT)


# ============================================================
# 4. MODEL / TRAINING PARAMETERS
# ============================================================

# key_dim is kept fixed at the number of EEG channels (19),
# matching the original architecture. Only the number of
# attention heads and the number of stacked transformer
# blocks are varied in this sweep.

KEY_DIM = 19

# Feed-forward network
FFN_DIM = 64

# Training parameters
EPOCHS = 20
BATCH_SIZE = 8

# Cross-validation parameters
N_SPLITS = 10

# Shuffle segments before splitting into folds
SHUFFLE_FOLDS = True

# Set to False to use plain KFold instead of StratifiedKFold
USE_STRATIFIED = True

# Inference-time measurement
# Attention memory grows as batch * num_heads * 2500^2 * 4 bytes,
# so keep this small. Using the training batch size is safe, since
# training needs more memory than inference. Lower it (e.g. 4 or 2)
# if you sweep to many heads and still hit ResourceExhaustedError.
INFERENCE_BATCH_SIZE = BATCH_SIZE
N_LATENCY_SAMPLES = 100       # segments used for batch-size-1 latency


# ------------------------------------------------------------
# CONFIGURATIONS TO COMPARE
#
# Each entry is one experiment: a given number of attention
# heads and a given number of stacked TransformerLayer blocks.
# The FULL 10-fold CV is run once per entry.
# ------------------------------------------------------------

CONFIGURATIONS = [
    {"num_heads": 4, "num_blocks": 1},
]


# ============================================================
# 5. BAND-PASS FILTER
# ============================================================

def butter_bandpass(lowcut, highcut, fs, order=2):
    """
    Design a Butterworth band-pass filter.
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype="band")

    return b, a


def apply_bandpass_filter(data, b, a):
    """
    Apply zero-phase Butterworth filtering.
    """

    return filtfilt(b, a, data, axis=-1)


# ============================================================
# 6. SEGMENTATION
# ============================================================

def segment_data(data, segment_length, sfreq):
    """
    Divide EEG recording into non-overlapping segments.
    """

    samples_per_segment = int(segment_length * sfreq)

    n_segments = data.shape[1] // samples_per_segment

    segments = []

    for i in range(n_segments):

        start = i * samples_per_segment
        end = start + samples_per_segment

        segment = data[:, start:end]

        if segment.shape[1] == samples_per_segment:
            segments.append(segment)

    return segments


# ============================================================
# 7. LOAD DATA  (NOT included in inference time)
# ============================================================

def load_and_segment_subjects(
    folder_path,
    lowcut,
    highcut,
    sfreq,
    segment_length,
    order=2
):
    """
    Load all EDF files. Each EDF file is treated as ONE
    participant.

    The 'subject_ids' array is still returned, but it is now
    used ONLY for reporting (per-participant accuracy), not
    for splitting the folds.
    """

    b, a = butter_bandpass(lowcut, highcut, sfreq, order)

    X = []
    y = []
    subject_ids = []

    subject_information = []

    edf_files = sorted(
        [
            f
            for f in os.listdir(folder_path)
            if f.lower().endswith(".edf")
        ]
    )

    print("\nNumber of EDF files found:", len(edf_files))

    subject_id = 0

    for filename in edf_files:

        filename_lower = filename.lower()

        if filename_lower.startswith("h"):

            label = 0
            class_name = "HC"

        elif filename_lower.startswith("s"):

            label = 1
            class_name = "SZ"

        else:

            print(f"\nSkipping {filename}")
            print("Reason: filename does not start with 'h' or 's'.")

            continue

        filepath = os.path.join(folder_path, filename)

        print("\n" + "=" * 65)
        print(f"Participant ID : {subject_id}")
        print(f"File           : {filename}")
        print(f"Class          : {class_name}")

        try:

            raw = mne.io.read_raw_edf(
                filepath,
                preload=True,
                verbose=False
            )

        except Exception as e:

            print(f"Could not read {filename}: {e}")
            continue

        data = raw.get_data()

        print("Original data shape:", data.shape)

        if data.shape[0] != 19:

            print("WARNING:")
            print(
                f"Expected 19 channels but found "
                f"{data.shape[0]} channels."
            )
            print("This participant will be skipped.")

            continue

        filtered_data = apply_bandpass_filter(data, b, a)

        # No StandardScaler here - normalisation happens
        # inside each fold, using training segments only.

        segments = segment_data(
            filtered_data,
            segment_length,
            sfreq
        )

        print("Number of complete segments:", len(segments))

        for segment in segments:

            # (19, 2500) -> (2500, 19)
            segment = segment.T

            X.append(segment.astype(np.float32))
            y.append(label)
            subject_ids.append(subject_id)

        subject_information.append(
            {
                "subject_id": subject_id,
                "filename": filename,
                "class": class_name,
                "label": label,
                "segments": len(segments)
            }
        )

        subject_id += 1

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int32)
    subject_ids = np.asarray(subject_ids, dtype=np.int32)

    subject_information = pd.DataFrame(subject_information)

    return X, y, subject_ids, subject_information


# ============================================================
# 8. LOAD DATASET
# ============================================================

X, y, subject_ids, subject_information = (
    load_and_segment_subjects(
        folder_path=folder_path,
        lowcut=LOWCUT,
        highcut=HIGHCUT,
        sfreq=SFREQ,
        segment_length=SEGMENT_LENGTH,
        order=FILTER_ORDER
    )
)


# ============================================================
# 9. DATASET SUMMARY
# ============================================================

print("\n")
print("=" * 65)
print("DATASET SUMMARY")
print("=" * 65)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of participants:", len(np.unique(subject_ids)))
print("Number of HC segments:", np.sum(y == 0))
print("Number of SZ segments:", np.sum(y == 1))

print("\nParticipant-level information:")
print(subject_information.to_string(index=False))


# ============================================================
# 10. ADJUST N_SPLITS IF NECESSARY
#
# With segment-wise CV the limit is the number of segments in
# the smallest class, not the number of participants.
# ============================================================

n_segments_total = X.shape[0]

min_class_count = int(min(np.sum(y == 0), np.sum(y == 1)))

max_allowed_splits = (
    min_class_count if USE_STRATIFIED else n_segments_total
)

if N_SPLITS > max_allowed_splits:

    print(
        "\nWARNING: N_SPLITS "
        f"({N_SPLITS}) is greater than the maximum allowed "
        f"({max_allowed_splits})."
    )

    print(f"Reducing N_SPLITS to {max_allowed_splits}.")

    N_SPLITS = max_allowed_splits


input_shape = (X.shape[1], X.shape[2])

print("\nInput shape:", input_shape)


# ============================================================
# 11. TRANSFORMER ENCODER BLOCK
# ============================================================

class TransformerLayer(Layer):
    """
    One Transformer encoder block.

    Input
      -> Multi-Head Self-Attention (num_heads, key_dim)
      -> Residual + LayerNorm
      -> Dense(ffn_dim, ReLU) -> Dense(key_dim)
      -> Residual + LayerNorm
    """

    def __init__(
        self,
        num_heads,
        key_dim,
        ffn_dim=64,
        block_index=0,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=key_dim,
            name=f"MultiHeadSelfAttention_{block_index}"
        )

        self.layer_norm1 = LayerNormalization(
            epsilon=1e-6,
            name=f"LayerNorm_1_{block_index}"
        )

        self.dense1 = Dense(
            ffn_dim,
            activation="relu",
            name=f"FFN_Dense_{ffn_dim}_{block_index}"
        )

        self.dense2 = Dense(
            key_dim,
            name=f"FFN_Dense_{key_dim}_{block_index}"
        )

        self.layer_norm2 = LayerNormalization(
            epsilon=1e-6,
            name=f"LayerNorm_2_{block_index}"
        )

    def call(self, inputs, training=None):

        attention_output = self.attention(
            query=inputs,
            key=inputs,
            value=inputs,
            training=training
        )

        x = self.layer_norm1(inputs + attention_output)

        ffn_output = self.dense1(x)
        ffn_output = self.dense2(ffn_output)

        output = self.layer_norm2(x + ffn_output)

        return output


# ============================================================
# 12. CREATE DeepNeuroTransFormer
#
# num_heads   : attention heads per transformer block
# num_blocks  : number of stacked transformer encoder blocks
# ============================================================

def create_transformer_model(
    input_shape,
    num_heads,
    num_blocks,
    key_dim=KEY_DIM,
    ffn_dim=FFN_DIM
):

    inputs = Input(shape=input_shape, name="EEG_Input")

    x = inputs

    # Stack N transformer encoder blocks. Each block keeps the
    # same (time, channels) shape, so blocks chain directly.

    for block_index in range(num_blocks):

        x = TransformerLayer(
            num_heads=num_heads,
            key_dim=key_dim,
            ffn_dim=ffn_dim,
            block_index=block_index,
            name=f"DeepNeuroTransformerBlock_{block_index}"
        )(x)

    x = GlobalAveragePooling1D(name="GlobalAveragePooling")(x)

    outputs = Dense(
        2,
        activation="softmax",
        name="Output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="DeepNeuroTransFormer"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ============================================================
# 13. TRAINING-ONLY STANDARDISATION  (NOT included in
#     inference time)
# ============================================================

def standardise_train_test(X_train, X_test):
    """
    Standardise EEG channels without leakage of the
    normalisation statistics.
    The StandardScaler is fit ONLY on X_train.
    """

    n_channels = X_train.shape[-1]

    train_2d = X_train.reshape(-1, n_channels)
    test_2d = X_test.reshape(-1, n_channels)

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(train_2d)
    test_scaled = scaler.transform(test_2d)

    X_train_scaled = train_scaled.reshape(X_train.shape)
    X_test_scaled = test_scaled.reshape(X_test.shape)

    return (
        X_train_scaled.astype(np.float32),
        X_test_scaled.astype(np.float32)
    )


# ============================================================
# 14. INFERENCE-TIME MEASUREMENT
#
# Times ONLY the model forward pass. The input is already a
# standardised tensor, so loading, filtering, segmentation,
# standardisation and host-to-tensor conversion are all
# outside the timed regions.
#
# A tf.function with a fixed input signature is used so that
# the graph is traced once (during warm-up) and not re-traced
# for different batch sizes (e.g. the smaller final batch).
# ============================================================

def measure_inference_time(
    model,
    X_test,
    input_shape,
    batch_size=INFERENCE_BATCH_SIZE,
    n_latency=N_LATENCY_SAMPLES
):
    """
    Returns:
        probabilities        : model outputs (numpy)
        total_time_s         : seconds for the whole test set (batched)
        ms_per_segment       : total_time_s / n_segments, in ms
        latency_ms_batch1    : mean ms per segment with batch size 1
    """

    @tf.function(
        input_signature=[
            tf.TensorSpec(
                shape=(None,) + tuple(input_shape),
                dtype=tf.float32
            )
        ]
    )
    def infer(batch):
        return model(batch, training=False)

    # Convert to tensor BEFORE timing starts
    X_test_tf = tf.convert_to_tensor(X_test, dtype=tf.float32)

    n_segments = int(X_test_tf.shape[0])

    # ---- Warm-up (graph tracing, kernel / device init) ----
    _ = infer(X_test_tf[:batch_size]).numpy()
    _ = infer(X_test_tf[:1]).numpy()

    # ---- (a) Batched inference on the whole test set ----
    start = time.perf_counter()

    outputs = []

    for i in range(0, n_segments, batch_size):
        outputs.append(infer(X_test_tf[i:i + batch_size]))

    # .numpy() forces completion (matters for async GPU execution)
    probabilities = tf.concat(outputs, axis=0).numpy()

    end = time.perf_counter()

    total_time_s = end - start
    ms_per_segment = (total_time_s / n_segments) * 1000

    # ---- (b) Single-segment latency (batch size = 1) ----
    n_latency = min(n_latency, n_segments)

    start = time.perf_counter()

    for i in range(n_latency):
        _ = infer(X_test_tf[i:i + 1]).numpy()

    end = time.perf_counter()

    latency_ms_batch1 = ((end - start) / n_latency) * 1000

    return (
        probabilities,
        total_time_s,
        ms_per_segment,
        latency_ms_batch1
    )


# ============================================================
# 15. RUN ONE FULL 10-FOLD CV FOR A GIVEN CONFIGURATION
#
# Segments are split with StratifiedKFold (or KFold).
# Participant identity is NOT used for splitting.
# ============================================================

def run_kfold_cv(
    num_heads,
    num_blocks,
    X,
    y,
    subject_ids,
    subject_information,
    input_shape,
    n_splits=N_SPLITS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    verbose_training=1
):
    """
    Runs the full segment-wise K-fold CV pipeline for one
    (num_heads, num_blocks) configuration and returns
    overall + fold-level + participant-level results,
    including inference time.
    """

    if USE_STRATIFIED:

        kf = StratifiedKFold(
            n_splits=n_splits,
            shuffle=SHUFFLE_FOLDS,
            random_state=seed if SHUFFLE_FOLDS else None
        )

    else:

        from sklearn.model_selection import KFold

        kf = KFold(
            n_splits=n_splits,
            shuffle=SHUFFLE_FOLDS,
            random_state=seed if SHUFFLE_FOLDS else None
        )

    fold_results = []

    all_true = []
    all_pred = []
    all_probabilities = []
    all_subject_ids = []

    print("\n")
    print("#" * 65)
    print(
        f"CONFIGURATION: num_heads={num_heads}, "
        f"num_blocks={num_blocks}"
    )
    print("#" * 65)

    for fold, (train_index, test_index) in enumerate(
        kf.split(X, y),
        start=1
    ):

        X_train = X[train_index]
        X_test = X[test_index]

        y_train = y[train_index]
        y_test = y[test_index]

        X_train, X_test = standardise_train_test(X_train, X_test)

        n_test_hc = int(np.sum(y_test == 0))
        n_test_sz = int(np.sum(y_test == 1))

        print("\n" + "=" * 65)
        print(
            f"[heads={num_heads}, blocks={num_blocks}] "
            f"FOLD {fold}/{n_splits}"
        )
        print(f"Training segments     : {len(train_index)}")
        print(f"Testing segments      : {len(test_index)}")
        print(f"Test HC / SZ segments : {n_test_hc} / {n_test_sz}")
        print("=" * 65)

        tf.keras.backend.clear_session()
        gc.collect()

        np.random.seed(seed + fold)
        random.seed(seed + fold)
        tf.random.set_seed(seed + fold)

        model = create_transformer_model(
            input_shape,
            num_heads=num_heads,
            num_blocks=num_blocks
        )

        # Training is NOT included in inference time
        history = model.fit(
            X_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            shuffle=True,
            verbose=verbose_training
        )

        # ----------------------------------------------------
        # Inference (timed: model forward pass only)
        # ----------------------------------------------------

        (
            probabilities,
            total_inference_time_s,
            inference_ms_per_segment,
            latency_ms_batch1
        ) = measure_inference_time(
            model,
            X_test,
            input_shape
        )

        y_pred = np.argmax(probabilities, axis=1)

        fold_accuracy = accuracy_score(y_test, y_pred)

        print(f"\nFold {fold} accuracy: {fold_accuracy * 100:.2f}%")

        print(
            f"Inference time (test set, {len(test_index)} segments): "
            f"{total_inference_time_s:.3f} s"
        )
        print(
            f"Inference time per segment (batched): "
            f"{inference_ms_per_segment:.3f} ms"
        )
        print(
            f"Single-segment latency (batch=1): "
            f"{latency_ms_batch1:.3f} ms"
        )

        all_true.extend(y_test.tolist())
        all_pred.extend(y_pred.tolist())
        all_probabilities.extend(probabilities.tolist())
        all_subject_ids.extend(subject_ids[test_index].tolist())

        fold_results.append(
            {
                "num_heads": num_heads,
                "num_blocks": num_blocks,
                "fold": fold,
                "n_train_segments": len(train_index),
                "n_test_segments": len(test_index),
                "n_test_HC": n_test_hc,
                "n_test_SZ": n_test_sz,
                "fold_accuracy": fold_accuracy,
                "total_inference_time_s": total_inference_time_s,
                "inference_ms_per_segment": inference_ms_per_segment,
                "latency_ms_batch1": latency_ms_batch1
            }
        )

        del model
        del history

        tf.keras.backend.clear_session()
        gc.collect()

    fold_results_df = pd.DataFrame(fold_results)

    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)
    all_probabilities = np.asarray(all_probabilities)
    all_subject_ids = np.asarray(all_subject_ids)

    # --------------------------------------------------------
    # Participant-wise accuracy (reporting only)
    #
    # Every segment appears exactly once as a test segment
    # across the folds, so this still aggregates cleanly per
    # participant - but the model DID see other segments from
    # the same participant during training.
    # --------------------------------------------------------

    participant_rows = []

    for subject_id in np.unique(all_subject_ids):

        subject_mask = (all_subject_ids == subject_id)

        subject_accuracy = accuracy_score(
            all_true[subject_mask],
            all_pred[subject_mask]
        )

        subject_row = subject_information[
            subject_information["subject_id"] == subject_id
        ].iloc[0]

        participant_rows.append(
            {
                "num_heads": num_heads,
                "num_blocks": num_blocks,
                "subject_id": int(subject_id),
                "filename": subject_row["filename"],
                "class": subject_row["class"],
                "n_test_segments": int(subject_mask.sum()),
                "accuracy": subject_accuracy
            }
        )

    participant_results_df = pd.DataFrame(participant_rows)

    # --------------------------------------------------------
    # Overall out-of-fold metrics
    # --------------------------------------------------------

    cm = confusion_matrix(all_true, all_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    overall_accuracy = accuracy_score(all_true, all_pred)

    precision = precision_score(all_true, all_pred, zero_division=0)

    sensitivity = recall_score(all_true, all_pred, zero_division=0)

    f1 = f1_score(all_true, all_pred, zero_division=0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    mean_fold_accuracy = fold_results_df["fold_accuracy"].mean()
    std_fold_accuracy = fold_results_df["fold_accuracy"].std()

    mean_subject_accuracy = participant_results_df["accuracy"].mean()
    std_subject_accuracy = participant_results_df["accuracy"].std()

    # --------------------------------------------------------
    # Inference-time statistics across folds
    # --------------------------------------------------------

    mean_total_inference_s = fold_results_df["total_inference_time_s"].mean()
    std_total_inference_s = fold_results_df["total_inference_time_s"].std()

    mean_inference_ms = fold_results_df["inference_ms_per_segment"].mean()
    std_inference_ms = fold_results_df["inference_ms_per_segment"].std()

    mean_latency_ms = fold_results_df["latency_ms_batch1"].mean()
    std_latency_ms = fold_results_df["latency_ms_batch1"].std()

    # --------------------------------------------------------
    # Trainable parameter count (model capacity for this config)
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    param_count_model = create_transformer_model(
        input_shape,
        num_heads=num_heads,
        num_blocks=num_blocks
    )

    n_params = param_count_model.count_params()

    del param_count_model

    tf.keras.backend.clear_session()
    gc.collect()

    print("\n")
    print("-" * 65)
    print(
        f"CONFIGURATION SUMMARY: "
        f"num_heads={num_heads}, num_blocks={num_blocks}"
    )
    print("-" * 65)
    print(f"Trainable parameters    : {n_params}")
    print(f"Overall accuracy        : {overall_accuracy * 100:.2f}%")
    print(f"Overall sensitivity     : {sensitivity * 100:.2f}%")
    print(f"Overall specificity     : {specificity * 100:.2f}%")
    print(f"Overall precision       : {precision * 100:.2f}%")
    print(f"Overall F1-score        : {f1 * 100:.2f}%")
    print(f"Mean fold accuracy      : {mean_fold_accuracy * 100:.2f}%")
    print(f"SD fold accuracy        : {std_fold_accuracy * 100:.2f}%")
    print(f"Mean subject accuracy   : {mean_subject_accuracy * 100:.2f}%")
    print(f"SD subject accuracy     : {std_subject_accuracy * 100:.2f}%")
    print(
        f"Test-set inference (s)  : "
        f"{mean_total_inference_s:.3f} ± {std_total_inference_s:.3f}"
    )
    print(
        f"Per segment, batched    : "
        f"{mean_inference_ms:.3f} ± {std_inference_ms:.3f} ms"
    )
    print(
        f"Per segment, batch=1    : "
        f"{mean_latency_ms:.3f} ± {std_latency_ms:.3f} ms"
    )

    summary_row = {
        "num_heads": num_heads,
        "num_blocks": num_blocks,
        "n_params": n_params,
        "overall_accuracy": overall_accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1,
        "mean_fold_accuracy": mean_fold_accuracy,
        "std_fold_accuracy": std_fold_accuracy,
        "mean_subject_accuracy": mean_subject_accuracy,
        "std_subject_accuracy": std_subject_accuracy,
        "mean_total_inference_s": mean_total_inference_s,
        "std_total_inference_s": std_total_inference_s,
        "mean_inference_ms_per_segment": mean_inference_ms,
        "std_inference_ms_per_segment": std_inference_ms,
        "mean_latency_ms_batch1": mean_latency_ms,
        "std_latency_ms_batch1": std_latency_ms
    }

    return {
        "summary_row": summary_row,
        "fold_results_df": fold_results_df,
        "participant_results_df": participant_results_df,
        "all_true": all_true,
        "all_pred": all_pred,
        "all_probabilities": all_probabilities,
        "all_subject_ids": all_subject_ids,
        "confusion_matrix": cm
    }


# ============================================================
# 16. RUN THE FULL SWEEP OVER CONFIGURATIONS
# ============================================================

summary_rows = []

all_fold_results = []
all_participant_results = []

detailed_results_by_config = {}

for config in CONFIGURATIONS:

    num_heads = config["num_heads"]
    num_blocks = config["num_blocks"]

    result = run_kfold_cv(
        num_heads=num_heads,
        num_blocks=num_blocks,
        X=X,
        y=y,
        subject_ids=subject_ids,
        subject_information=subject_information,
        input_shape=input_shape,
        n_splits=N_SPLITS,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        seed=SEED,
        verbose_training=1
    )

    summary_rows.append(result["summary_row"])

    all_fold_results.append(result["fold_results_df"])
    all_participant_results.append(result["participant_results_df"])

    config_key = f"heads{num_heads}_blocks{num_blocks}"

    detailed_results_by_config[config_key] = result

    # Save per-configuration segment-level predictions

    prediction_results = pd.DataFrame(
        {
            "subject_id": result["all_subject_ids"],
            "true_label": result["all_true"],
            "predicted_label": result["all_pred"],
            "probability_HC": result["all_probabilities"][:, 0],
            "probability_SZ": result["all_probabilities"][:, 1]
        }
    )

    prediction_results["true_class"] = prediction_results[
        "true_label"
    ].map({0: "HC", 1: "SZ"})

    prediction_results["predicted_class"] = prediction_results[
        "predicted_label"
    ].map({0: "HC", 1: "SZ"})

    prediction_results.to_csv(
        f"DeepNeuroTransFormer_predictions_{config_key}.csv",
        index=False
    )


# ============================================================
# 17. COMPARISON TABLE ACROSS CONFIGURATIONS
# ============================================================

comparison_df = pd.DataFrame(summary_rows)

comparison_df = comparison_df.sort_values(
    by="overall_accuracy",
    ascending=False
).reset_index(drop=True)

print("\n\n")
print("=" * 80)
print("COMPARISON ACROSS ATTENTION-HEAD / TRANSFORMER-BLOCK CONFIGURATIONS")
print("=" * 80)

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "overall_accuracy": "{:.4f}".format,
            "sensitivity": "{:.4f}".format,
            "specificity": "{:.4f}".format,
            "precision": "{:.4f}".format,
            "f1_score": "{:.4f}".format,
            "mean_fold_accuracy": "{:.4f}".format,
            "std_fold_accuracy": "{:.4f}".format,
            "mean_subject_accuracy": "{:.4f}".format,
            "std_subject_accuracy": "{:.4f}".format,
            "mean_total_inference_s": "{:.3f}".format,
            "std_total_inference_s": "{:.3f}".format,
            "mean_inference_ms_per_segment": "{:.3f}".format,
            "std_inference_ms_per_segment": "{:.3f}".format,
            "mean_latency_ms_batch1": "{:.3f}".format,
            "std_latency_ms_batch1": "{:.3f}".format
        }
    )
)

best_config_row = comparison_df.iloc[0]

print("\nBest configuration by overall out-of-fold accuracy:")
print(
    f"  num_heads = {int(best_config_row['num_heads'])}, "
    f"num_blocks = {int(best_config_row['num_blocks'])}, "
    f"accuracy = {best_config_row['overall_accuracy'] * 100:.2f}%"
)


# ============================================================
# 18. INFERENCE-TIME COMPARISON TABLE
#     (model forward pass only; excludes data loading,
#      filtering, segmentation and standardisation)
# ============================================================

inference_cols = [
    "num_heads",
    "num_blocks",
    "n_params",
    "mean_total_inference_s",
    "std_total_inference_s",
    "mean_inference_ms_per_segment",
    "std_inference_ms_per_segment",
    "mean_latency_ms_batch1",
    "std_latency_ms_batch1"
]

inference_df = comparison_df[inference_cols].copy()

print("\n\n")
print("=" * 80)
print("INFERENCE TIME (model forward pass only)")
print("=" * 80)

print(
    inference_df.to_string(
        index=False,
        formatters={
            "mean_total_inference_s": "{:.3f}".format,
            "std_total_inference_s": "{:.3f}".format,
            "mean_inference_ms_per_segment": "{:.3f}".format,
            "std_inference_ms_per_segment": "{:.3f}".format,
            "mean_latency_ms_batch1": "{:.3f}".format,
            "std_latency_ms_batch1": "{:.3f}".format
        }
    )
)

print("\nDevices:", tf.config.list_physical_devices())




Samples per segment: 2500

Number of EDF files found: 28

Participant ID : 0
File           : h01.edf
Class          : HC
Original data shape: (19, 231250)
Number of complete segments: 92

Participant ID : 1
File           : h02.edf
Class          : HC
Original data shape: (19, 227500)
Number of complete segments: 91

Participant ID : 2
File           : h03.edf
Class          : HC
Original data shape: (19, 227500)
Number of complete segments: 91

Participant ID : 3
File           : h04.edf
Class          : HC
Original data shape: (19, 231250)
Number of complete segments: 92

Participant ID : 4
File           : h05.edf
Class          : HC
Original data shape: (19, 236250)
Number of complete segments: 94

Participant ID : 5
File           : h06.edf
Class          : HC
Original data shape: (19, 232500)
Number of complete segments: 93

Participant ID : 6
File           : h07.edf
Class          : HC
Original data shape: (19, 227500)
Number of complete segments: 91

Participant ID : 7
File  